# 02 — Bronze Layer (Auto Loader → VARIANT)

Incrementally loads GUMBO JSON from the Volume landing zone into a single
VARIANT-based bronze Delta table.

## Why this matters — Well-Architected Framework

| WAF pillar | What this notebook does, and why it's good |
|------------|---------------------------------------------|
| **Reliability** | Auto Loader checkpoints mean "already processed" is remembered on disk — re-runs only pick up *new* files. The `_rescued` column captures any schema drift so MLB changing their API shape never silently drops data. |
| **Performance Efficiency** | Liquid clustering on `file_batch_time` co-locates the newest files physically — silver-layer jobs that say `WHERE file_batch_time > last_run` scan kilobytes, not gigabytes. |
| **Data Governance** | `VARIANT` preserves the exact raw payload forever (plus source-file metadata) — full auditability, and you can reprocess at any time without going back to the API. |
| **Operational Excellence** | `trigger(availableNow=True)` makes this a batch job you can schedule — starts, processes what's new, stops. No streaming-cluster babysitting. |

> **Why not `MERGE`?** `MERGE` isn't supported through Spark Connect. We don't need it: bronze
> is append-only by design, and Auto Loader gives us exactly-once appends via checkpoints.


In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/raw_gumbo"

print(f"Bronze: {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Source volume: {VOLUME_PATH}")


Bronze: alexander_booth.mlb_gumbo_waf_bronze
Source volume: /Volumes/alexander_booth/mlb_gumbo_waf_bronze/raw_gumbo


## Create the bronze table

> **WAF — Performance Efficiency.** `CLUSTER BY (file_batch_time)` is the
> single most important line in this DDL. Liquid clustering adapts file
> layout to the predicate shape your silver job uses; no manual OPTIMIZE
> ZORDER choreography required.

In [2]:
bronze_table = f"{UC_CATALOG}.{BRONZE_SCHEMA}.raw_gumbo"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {bronze_table} (
        data                   VARIANT   COMMENT 'Raw MLB GUMBO JSON payload, stored as VARIANT for shredded query access',
        source_file            STRING    COMMENT 'Path to the source JSON file in the landing Volume',
        source_file_name       STRING    COMMENT 'File name only, e.g. game_data_<gamePk>.json',
        source_file_size       BIGINT    COMMENT 'Size of the source file in bytes',
        file_modification_time TIMESTAMP COMMENT 'Last-modified timestamp from the object store',
        file_batch_time        TIMESTAMP COMMENT 'Wall-clock time Auto Loader landed this row'
    )
    USING DELTA
    CLUSTER BY (file_batch_time)
    COMMENT 'Bronze — raw MLB GUMBO JSON with ingest-time metadata. Re-runnable via Auto Loader checkpoints.'
""")
print(f"Bronze table ready: {bronze_table}")


Bronze table ready: alexander_booth.mlb_gumbo_waf_bronze.raw_gumbo


## Run Auto Loader (availableNow = batch mode)

> **WAF — Reliability.** Three lines do the heavy lifting:
> 1. `cloudFiles.schemaLocation` — schema is remembered between runs.
> 2. `rescuedDataColumn = "_rescued"` — a safety net for upstream schema drift: new fields land in a rescue column, we never drop data.
> 3. `checkpointLocation` — the incremental state. Delete the checkpoint to force a full re-ingest; keep it to process only new files.

In [3]:
checkpoint_path = f"{VOLUME_PATH}/_checkpoints/bronze_raw_gumbo"
schema_path     = f"{VOLUME_PATH}/_schemas/bronze_raw_gumbo"

# Read each file as a single blob (wholetext) so we can PARSE_JSON the entire
# raw JSON document cleanly into a VARIANT. Using format=json with multiLine
# and then TO_JSON(STRUCT(*)) would serialize nested objects inconsistently —
# wholetext + PARSE_JSON gives us a clean, fully-shredded VARIANT.
query = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("wholetext", "true")
        .option("cloudFiles.schemaLocation", schema_path)
        .load(VOLUME_PATH)
        .selectExpr(
            "PARSE_JSON(value) AS data",
            "_metadata.file_path              AS source_file",
            "_metadata.file_name              AS source_file_name",
            "_metadata.file_size              AS source_file_size",
            "_metadata.file_modification_time AS file_modification_time",
            "current_timestamp()              AS file_batch_time",
        )
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(bronze_table)
)

query.awaitTermination()
print("Bronze load complete.")


Bronze load complete.


## Verify

In [4]:
rows = spark.table(bronze_table).count()
print(f"{bronze_table}: {rows:,} rows")

spark.sql(f"""
    SELECT source_file_name,
           data:gamePk::int AS game_pk,
           data:gameData:datetime:officialDate::date AS official_date,
           data:gameData:teams:home:name::string AS home_team,
           data:gameData:teams:away:name::string AS away_team,
           file_batch_time
    FROM {bronze_table}
    ORDER BY file_batch_time DESC
    LIMIT 5
""").show(truncate=False)


alexander_booth.mlb_gumbo_waf_bronze.raw_gumbo: 2,477 rows


+---------------------+-------+-------------+---------------------+--------------------+-----------------------+
|source_file_name     |game_pk|official_date|home_team            |away_team           |file_batch_time        |
+---------------------+-------+-------------+---------------------+--------------------+-----------------------+
|game_data_777915.json|777915 |2025-05-14   |Seattle Mariners     |New York Yankees    |2026-04-21 20:52:47.776|
|game_data_778176.json|778176 |2025-04-25   |Los Angeles Dodgers  |Pittsburgh Pirates  |2026-04-21 20:52:47.776|
|game_data_778304.json|778304 |2025-04-16   |Cincinnati Reds      |Seattle Mariners    |2026-04-21 20:52:47.776|
|game_data_778376.json|778376 |2025-04-09   |San Francisco Giants |Cincinnati Reds     |2026-04-21 20:52:47.776|
|game_data_778313.json|778313 |2025-04-15   |Philadelphia Phillies|San Francisco Giants|2026-04-21 20:52:47.776|
+---------------------+-------+-------------+---------------------+--------------------+--------